In [ ]:
import os
if not os.path.exists("/content/model"):
  !git clone https://github.com/ggarciabaez/VLA.git
  %cd VLA
  !git checkout qformer
  %mv model ../
  %mv scripts/train_datasets.py ../
  %mv scripts/train_tools.py ../
  %cd ../
  %rm -r VLA
  !pip install --upgrade sentencepiece transformers==5.5.4 tqdm onnx wandb

In [ ]:
from google.colab import drive, runtime
import pickle
from train_tools import *
from train_datasets import EmbeddingDataset
import wandb

# Authenticate to Weights & Biases
# You can set your API key in Colab Secrets (key: 'WANDB_API_KEY') or enter it interactively
try:
    from google.colab import userdata
    api_key = userdata.get('WANDB_API_KEY')
    wandb.login(key=api_key)
except Exception:
    wandb.login()

drive.mount("/content/drive")

SRC = Path("/content/drive/MyDrive/VLA/preencoded/mt10_grip/")
DST = Path("/content/data/")
DST.mkdir(parents=True, exist_ok=True)
use_pkl = False

if use_pkl:
    p = SRC.joinpath("ds.pkl")
    if not p.exists():
        download_shards(SRC, DST)
        use_pkl = False
    else:
        print("Found pickled dataset. Now opening...")
        with open(p, "rb") as f:
            dataset = pickle.load(f)
            print(f"Loaded dataset from pickle: {p}")
else:
    download_shards(SRC, DST)

In [1]:
import time
import torch.nn as nn
from torch.amp import GradScaler
from model.vla import VLA
from model.utils import VLAConfig, freeze_except_last_n_layers

In [2]:
CFG = dict(
    # Paths
    data_dir          = "/content/data/",
    episode_glob      = "ep*.npz",
    checkpoint_dir    = "/content/checkpoints",
    save_path         = "/content/drive/MyDrive/VLA/checkpoints/model_mt10",

    # Values you're likely to modify
    epochs            = 30,
    batch_size        = 128,     # tasks per forward pass
    val_batch_size    = 32,      # To get more iterations
    resume            = False,

    # Training
    seed              = 42,
    val_split         = 0.1,    # fraction of episodes held out (not tasks)
    optimizer_stride  = 8,      # gradient accumulation over N timesteps
    grad_clip_norm    = 1.0,
    warmup_epochs     = 1.0,
    learning_rate     = 1e-3,
    backbone_lr_scale = 0.1,
    weight_decay      = 1e-2,
    log_every         = 100,    # optimizer steps between log lines

    # Model
    siglip_model_id   = "google/siglip2-base-patch16-224",
    dropout           = 0.1,
    n_heads           = 8,
    n_layers          = 6,
    n_action_layers   = 6,
    lq_size           = 64,
    state_dim         = 21,      # + xyz coords of item
    action_dim        = 4,
    chunk_size        = 16,
    flow_steps        = 10,
    img_size          = 224,

    # WandB Logging
    use_wandb         = True,
    wandb_project     = "VLA",
    wandb_entity      = None,   # set to your wandb entity/team if applicable
    wandb_run_name    = None,   # optional custom run name
    wandb_notes       = None,
    wandb_mode        = "online", # "online", "offline", or "disabled"
    watch_model       = True,  # whether to track model gradients with wandb.watch
)

In [5]:
# ── Training loop (variable-length, ContinuousDataset) ──────────────────────────
# No shared step_idx across a batch — every sample is an independent flat
# (col, t) pair, shuffled like an ordinary supervised dataset. optimizer_stride
# now means "accumulate over N mini-batches" rather than "N timesteps of one
# episode".

def run_epoch(
    model: VLA,
    dataset: EmbeddingDataset,
    mon: Monitor | None,
    sample_indices: list[int],
    optimizer,
    scheduler,
    scaler: GradScaler,
    device: torch.device,
    amp_dtype: torch.dtype,
    cfg: dict,
    epoch: int,
    global_step: int,
    train: bool,
) -> tuple[float, int]:
    model.train(mode=train)
    torch.set_grad_enabled(train)

    phase = "train" if train else "val"
    batch_size = cfg["batch_size"] if train else cfg["val_batch_size"]
    opt_stride = cfg["optimizer_stride"]
    log_every = cfg["log_every"]

    epoch_meter = AverageMeter()
    log_meter = AverageMeter()
    t0 = time.perf_counter()

    order = list(sample_indices)
    if train:
        random.shuffle(order)
    n_batches = math.ceil(len(order) / batch_size)

    if train:
        optimizer.zero_grad(set_to_none=True)

    for batch_idx, i in enumerate(tqdm(range(0, len(order), batch_size))):
        # TODO: suboptimal, delivered in dict. Could be better directly slicing.
        batch = order[i : i + batch_size]
        B = len(batch)

        # Each dataset item is a dictionary.
        batch_items = [dataset[j] for j in batch]

        # Pre-encoded SigLIP vision embeddings: (B, 196, 768)
        img_t = torch.stack(
            [item["images"] for item in batch_items]
        ).to(device, non_blocking=True)

        # Already normalized by ContinuousDataset: (B, state_dim)
        state_t = torch.stack(
            [item["states"] for item in batch_items]
        ).to(device, non_blocking=True)

        # Already normalized action chunks: (B, chunk_size, action_dim)
        action_t = torch.stack(
            [item["actions"] for item in batch_items]
        ).to(device, non_blocking=True)

        # Select one random prompt variant, shared across the mini-batch.
        n_variants = batch_items[0]["text_embeddings"].shape[0]
        variant = random.randrange(n_variants)

        # Pre-encoded text embeddings: (B, seq_len, text_embed_dim)
        txt = torch.stack([
            item["text_embeddings"][variant]
            for item in batch_items
        ]).to(device, non_blocking=True)

        # Corresponding attention masks: (B, seq_len)
        txt_mask = torch.stack([
            item["text_masks"][variant]
            for item in batch_items
        ]).to(device, non_blocking=True)

        with torch.autocast(
            device_type=device.type, dtype=amp_dtype, enabled=device.type == "cuda"
        ):
            loss = model.loss_preenc(action_t, img_t, txt, state_t, txt_mask)

        if not torch.isfinite(loss):
            print(
                f"[{phase}] non-finite loss at epoch {epoch} batch {batch_idx} — skipping"
            )
            if train:
                optimizer.zero_grad(set_to_none=True)
            continue

        if train:
            scaler.scale(loss / opt_stride).backward()

            should_step = ((batch_idx + 1) % opt_stride == 0) or (
                batch_idx + 1 == n_batches
            )
            if should_step:
                scaler.unscale_(optimizer)
                if mon is not None: mon.pre_clip(global_step)
                gn = nn.utils.clip_grad_norm_(model.parameters(), cfg["grad_clip_norm"])

                if global_step == 0:
                    check_gradient_flow(model)

                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                scheduler.step()
                if mon is not None: mon.post_step(global_step, gn)
                global_step += 1

                if global_step % log_every == 0:
                    if mon is not None: mon.run_probe(global_step)
                    elapsed = max(time.perf_counter() - t0, 1e-6)
                    sps = log_every * B * opt_stride / elapsed
                    current_lr = optimizer.param_groups[0]["lr"]
                    print(
                        f"epoch {epoch:>3d}  step {global_step:>7d}  "
                        f"loss {log_meter.avg:.4f}  "
                        f"lr {current_lr:.2e}  "
                        f"{sps:,.0f} samples/s"
                    )
                    if cfg.get("use_wandb", True) and wandb.run is not None:
                        wandb.log(
                            {
                                "train/step_loss": log_meter.avg,
                                "train/lr": current_lr,
                                "train/samples_per_sec": sps,
                                "epoch": epoch,
                            },
                            step=global_step,
                        )
                    log_meter.reset()
                    t0 = time.perf_counter()

        epoch_meter.update(loss.item(), B)
        log_meter.update(loss.item(), B)

    torch.set_grad_enabled(True)
    return epoch_meter.avg, global_step

In [3]:
def build_model_cfg(dataset, cfg: dict) -> VLAConfig:
    return VLAConfig(
        siglip_model_id = cfg["siglip_model_id"],
        n_trainable     = 0,
        dropout         = cfg["dropout"],
        d_model         = 768,
        n_heads         = cfg["n_heads"],
        n_layers        = cfg["n_layers"],
        n_action_layers = cfg["n_action_layers"],
        lq_size         = cfg["lq_size"],
        state_dim       = cfg["state_dim"],
        action_dim      = cfg["action_dim"],
        chunk_size      = cfg["chunk_size"],
        flow_steps      = cfg["flow_steps"],
        img_size        = cfg["img_size"],
        action_mean     = dataset.action_mean.tolist(),
        action_std      = dataset.action_std.tolist(),
        state_mean      = dataset.state_mean.tolist(),
        state_std       = dataset.state_std.tolist(),
    )


In [ ]:
def train(dataset, cfg: dict, run=None) -> None:
    seed_everything(cfg["seed"])
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device   : {device}")
    train_samples, val_samples = dataset.make_split(cfg["val_split"], cfg["seed"])

    model_cfg = build_model_cfg(dataset, cfg)
    model = VLA(model_cfg).to(device)
    if run is not None and cfg.get("watch_model", False):
        wandb.watch(model, log="gradients", log_freq=cfg.get("log_every", 100))
    optimizer = build_optimizer(
        model, cfg["learning_rate"], cfg["backbone_lr_scale"], cfg["weight_decay"]
    )

    # Steps per epoch = number of optimizer steps when processing all train samples
    n_batches_per_epoch = math.ceil(len(train_samples) / cfg["batch_size"])
    steps_per_epoch = math.ceil(n_batches_per_epoch / cfg["optimizer_stride"])
    scheduler = build_scheduler(optimizer, cfg, steps_per_epoch)

    bf16_ok = device.type == "cuda" and torch.cuda.is_bf16_supported()
    amp_dtype = torch.bfloat16 if bf16_ok else torch.float16
    scaler = GradScaler(
        device=device.type, enabled=(device.type == "cuda" and not bf16_ok)
    )
    print(f"AMP      : {amp_dtype}")

    TASKS = ["reach-v3", "push-v3", "pick-place-v3", "door-open-v3", "drawer-open-v3",
         "drawer-close-v3", "button-press-topdown-v3", "peg-insert-side-v3",
         "window-open-v3", "window-close-v3"]

    def build_probe(ds, device, variant=0, episode=0):
        col_of = {ds.task_names[c]: c for c in ds.episode_cols[episode]}   # one column per task
        sample_of = {s: i for i, s in enumerate(ds.samples)}               # (col, t) -> sample index
        batch = [ds[sample_of[(col_of[name], 0)]] for name in TASKS]
        probe = {
            "img":    torch.stack([b["images"] for b in batch]),                       # (10, 196, 768)
            "txt":    torch.stack([b["text_embeddings"][variant] for b in batch]),     # (10, 64, 768)
            "mask":   torch.stack([b["text_masks"][variant] for b in batch]),          # (10, 64)
            "state":  torch.stack([b["states"] for b in batch]),                       # (10, 21)
            "action": torch.stack([b["actions"] for b in batch]),                      # (10, chunk, 4)
        }
        return {k: v.to(device) for k, v in probe.items()}

    probe = build_probe(dataset, device)
    pair = [TASKS.index("reach-v3"), TASKS.index("pick-place-v3")]
    mon = Monitor(model, probe, pair)

    ckpt_dir = Path(cfg["checkpoint_dir"])
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    latest_path = ckpt_dir / "latest.pt"
    best_path = ckpt_dir / "best.pt"

    start_epoch = global_step = 0
    best_val = float("inf")

    if cfg["resume"] and best_path.exists():
        start_epoch, global_step, best_val = load_checkpoint(
            best_path, model, optimizer, scheduler, scaler
        )
        start_epoch += 1
        if cfg["freeze_encoders_resume"]:
            model.vision_encoder.backbone = freeze_except_last_n_layers(
                model.vision_encoder.backbone, 0
            )
            model.text_encoder.backbone = freeze_except_last_n_layers(
                model.text_encoder.backbone, 0, model_type="text"
            )
        print(
            f"Resumed  : epoch {start_epoch}, step {global_step}, best_val {best_val:.4f}, frozen heads {cfg['freeze_encoders_resume']}"
        )
    mon.run_probe(0)
    for epoch in range(start_epoch, cfg["epochs"]):
        train_loss, global_step = run_epoch(
            model,
            dataset,
            mon,
            train_samples,
            optimizer,
            scheduler,
            scaler,
            device,
            amp_dtype,
            cfg,
            epoch,
            global_step,
            train=True,
        )
        val_loss, _ = run_epoch(
            model,
            dataset,
            None,
            val_samples,
            None,
            None,
            scaler,
            device,
            amp_dtype,
            cfg,
            epoch,
            global_step,
            train=False,
        )

        is_best = val_loss < best_val
        if is_best:
            best_val = val_loss
            save_checkpoint(
                best_path,
                model,
                optimizer,
                scheduler,
                scaler,
                epoch,
                global_step,
                best_val,
            )

        current_lr = optimizer.param_groups[0]["lr"] if optimizer is not None else 0.0
        print(
            f"epoch {epoch:>3d}  "
            f"train {train_loss:.4f}  "
            f"val {val_loss:.4f}  "
            f"best {best_val:.4f}  "
            f"lr {current_lr:.2e}"
        )

        if cfg.get("use_wandb", True) and wandb.run is not None:
            metrics = {
                "train/epoch_loss": train_loss,
                "val/epoch_loss": val_loss,
                "val/best_loss": best_val,
                "train/lr": current_lr,
                "epoch": epoch,
            }
            wandb.log(metrics, step=global_step)
            if is_best:
                wandb.run.summary["best_val_loss"] = best_val
                wandb.run.summary["best_epoch"] = epoch

    if cfg.get("use_wandb", True) and wandb.run is not None:
        wandb.run.summary["final_train_loss"] = train_loss
        wandb.run.summary["final_val_loss"] = val_loss
        wandb.run.summary["total_global_steps"] = global_step

In [8]:
if not use_pkl:
    dataset = EmbeddingDataset(CFG)
    p = SRC.joinpath("ds.pkl")
    if not p.exists():
      with open(p, "wb") as f:
          pickle.dump(dataset, f)

Dataset: 20 episodes, 200,000 samples


In [ ]:
if CFG.get("use_wandb", True):
    run = wandb.init(
        project=CFG.get("wandb_project", "VLA"),
        entity=CFG.get("wandb_entity", None),
        name=CFG.get("wandb_run_name", None),
        notes=CFG.get("wandb_notes", None),
        mode=CFG.get("wandb_mode", "online"),
        config=CFG,
        reinit=True,
    )
    try:
        train(dataset, CFG, run=run)
    finally:
        wandb.finish()
else:
    train(dataset, CFG, run=None)

In [ ]:
import shutil
os.makedirs(CFG["save_path"], exist_ok=True)
shutil.copytree(CFG["checkpoint_dir"], CFG["save_path"], dirs_exist_ok=True)
time.sleep(120)  # wait 5m for everything to finish up
runtime.unassign()